In [1]:
from unstructured.partition.pdf import partition_pdf
from IPython.display import Image, display
import base64
import pandas as pd
import io
import tabulate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv
from collections import defaultdict
import json
import psycopg2
from pgvector.psycopg2 import register_vector


c:\Rayn\work\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [ ]:
# gemini embedding 2 preview is layout aware and can recognise table markdown as a relational grid. 
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

In [ ]:
chunks = partition_pdf(
    filename="government-data-security-policies.pdf",
    strategy="hi_res", # table detection and extraction as text_as_html

    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=True, # extracts images and tables as base64 encoded strings

    chunking_strategy="by_title",
    max_characters=2000,
    combine_text_under_n_chars=1000,
    new_after_n_chars=1500
)
'''chunk by title chunks by section headings and titles. For structured documents, each document
element is already semantically sound. computaionally cheaper than semantic chunking which is more 
suitable for highly unstructured data. 
'''

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 4803.53it/s]


In [ ]:
string = chunks[20].metadata.orig_elements[0].metadata.image_base64
display(Image(data=base64.b64decode(string)))

In [ ]:
'''loops through each element in chunk. If element is table, converts text_as_html to markdown
which is generally better (lower token cost, more readable to llms)
'''
def modify_text(chunk):
    text = []
    for element in chunk.metadata.orig_elements:
        if element.category == "Table" and getattr(element.metadata, "text_as_html", None):
            df = pd.read_html(io.StringIO(element.metadata.text_as_html))[0]
            markdown = df.to_markdown(index=False)
            text.append(markdown)
        else:
            text.append(element.text)
    chunk.text = "\n\n".join(text)

In [8]:
def extract_coords(chunk):
    highlights = []
    orig_elements = chunk.metadata.orig_elements
    for element in orig_elements:
        page = element.metadata.page_number
        coords = element.metadata.coordinates.points

        width = element.metadata.coordinates.system.width
        height = element.metadata.coordinates.system.height

        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
    
        clean_box = [
            round(max(0, float(min(xs))),2),
            round(max(0, float(min(ys))),2),
            round(min(width, float(max(xs))), 2),
            round(min(height, float(max(ys))), 2)
        ]
        highlights.append({
            "page": int(page),
            "bbox": clean_box,
            "width": int(width),
            "height": int(height)
        })
    setattr(chunk.metadata, "highlights", highlights)

In [18]:
def embed_text(chunk):
    vector = embeddings.embed_query(chunk.text)
    setattr(chunk.metadata, "vector", vector)

In [19]:
def process_chunk(chunk):
    modify_text(chunk)
    embed_text(chunk)
    extract_coords(chunk)

In [20]:
for chunk in chunks:
    process_chunk(chunk)

In [13]:
import fitz
def visualize_chunk_highlights(pdf_path, chunk, output_path="preview.pdf"):
    doc = fitz.open(pdf_path)
    highlights = getattr(chunk.metadata, "highlights", [])

    for h in highlights:
        page_index = h['page'] - 1
        page = doc[page_index]
        
        # 1. Get the page size as PyMuPDF sees it (72 DPI)
        actual_w = page.rect.width
        actual_h = page.rect.height
        
        # 2. Get the original system size from your metadata
        orig_w = h['width']
        orig_h = h['height']
        
        # 3. SCALE the coordinates
        # Math: (Raw_Value / Original_System_Size) * Actual_PDF_Size
        raw_bbox = h['bbox'] # [x1, y1, x2, y2]
        
        scaled_bbox = [
            (raw_bbox[0] / orig_w) * actual_w, # x_min
            (raw_bbox[1] / orig_h) * actual_h, # y_min
            (raw_bbox[2] / orig_w) * actual_w, # x_max
            (raw_bbox[3] / orig_h) * actual_h  # y_max
        ]
        
        # 4. Draw the correctly scaled rectangle
        rect = fitz.Rect(scaled_bbox)
        page.draw_rect(rect, color=(1, 1, 0), fill=(1, 1, 0), overlay=True, width=0)

    doc.save(output_path)
visualize_chunk_highlights("government-data-security-policies.pdf", chunks[1], output_path="preview.pdf")

In [16]:
print(chunks[20].text)

| 0                       | 1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
|:------------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------

In [21]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [23]:
chunk = chunks[0]
chunk_embedding = chunk.metadata.vector
chunk_text = chunk.text
if chunk.metadata.image_base64:
    image_base64 = base64.b64decode(chunk.metadata.image_base64)
else: 
    image_base64 = None
highlights = chunk.metadata.highlights

In [24]:
try:
    cur.execute('''
        INSERT INTO chunk_store (chunk_embedding, chunk_text, image_base64, highlights)
        VALUES (%s, %s, %s, %s)
    ''', (chunk_embedding, chunk_text, image_base64, json.dumps(highlights)))
    conn.commit()
except: 
    print("Error")
    conn.rollback()

In [27]:
def store_chunk(chunk):
    chunk_embedding = chunk.metadata.vector
    chunk_text = chunk.text
    if chunk.metadata.image_base64:
        image_base64 = base64.b64decode(chunk.metadata.image_base64)
    else: 
        image_base64 = None
    highlights = chunk.metadata.highlights
    try:
        cur.execute('''
            INSERT INTO chunk_store (chunk_embedding, chunk_text, image_base64, highlights)
            VALUES (%s, %s, %s, %s)
        ''', (chunk_embedding, chunk_text, image_base64, json.dumps(highlights)))
        conn.commit()
    except: 
        print("Error")
        conn.rollback()

In [33]:
cur.execute("SELECT chunk_text FROM chunk_store WHERE id = 26")
data = cur.fetchall()
print(data[0][0])



| 0                                          | 1                                                                                                                                                                                                                                                                                                      |
|:-------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Endpoint devices                           | Endpoint Devices refer to end-user devices designed for individual use (can be used by one or more users), Such as personal computers, mobile devices IP phones used to store, process or access Government data.                                                      

In [28]:
for chunk in chunks:
    store_chunk(chunk)

In [46]:
query = "data security risk management"
query_vector = embeddings.embed_query(query)

In [47]:
search_query = '''
    SELECT chunk_text, 1 - (chunk_embedding <=> %s) AS similarity
    FROM chunk_store
    ORDER BY chunk_embedding <=> %s
    LIMIT 5;
'''
try:
    cur.execute(search_query, (str(query_vector), str(query_vector)))
    data = cur.fetchall()
    df = pd.DataFrame(data)
except Exception as e:
    print(f"An error occurred: {e}")
    conn.rollback()
    

In [48]:
df.head()


,0,1
0,Section 2: Technical and process measures to p...,0.712063
1,The Government takes its responsibility as a c...,0.696333
2,Government Data Security Policies | 8\n\n\n\nS...,0.695529
3,Government Data Security Policies | 2\n\n\n\nS...,0.689794
4,\n\nTechnical measures and the complementary p...,0.675025


In [52]:
print(df[0][3])

Government Data Security Policies | 2



Section 1: Data Security Risk Management

01 To ensure adequate and eﬀective data security risk management, Agencies should perform data security risk assessments for their datasets, as part of the Government ICT Risk Management Methodology.

This will enable Agencies to identify data security risks, evaluate the risks, implement measures to mitigate the risks, assess the eﬀectiveness of the implemented measures and manage the risks within limits acceptable to the Agency.

02 Agencies should use the Data Security Risk Assessment Methodology, which is part of the Government ICT Risk Management Methodology, to conduct data security risk assessments for their datasets.

Agencies should conduct a data security risk assessment:

03

a) When acquiring a new dataset;

b) Developing a new ICT system that contains personal or entity data; or

c) When existing data which has not been risk assessed is ﬁrst used.

04

Agencies should review the data securit

In [53]:
cur.close()
conn.close()